# Assignment 4: Strava Data Report

## Rule et al's ten rules for computational analyses
Here is an outline of the rules that I followed in the notebook.  This was an exploratory analysis of a data sets that I was not familiar with so we did the following
- Rule 1: Tell a story for the audience.  I do this by making assumptions about the data and then talking about how those are good or bad and how that leads me to change my analysis
- Rule 2: Document the process, not just the results.  I have explainations in my code that outline what it is doing throughout the notebook
- Rule 3: Use Cell divisions to make steps clear.  The notebooke is divided into sections and contains markdown headers between code blocks
- Rule 4: Modularize code.  Some of the code is modularized, like the last section.  This is done for dynamic anaysis, while the rest of the notebook was more exploratory which is why it does not use modularized data
- Rule 5: Record dependencies.  I record these in the requirements.txt file
- Rule 6: User version control.  The workbook will be inside github repo
- Rule 9: Design your notebooks to be read, run, and explored.  The notebook is set up to be run top to bottom.  There is a local data set, but that is because that is how I was able to get this to work in Github.

## Setup and Dependencies

In [1]:
# import the dependencies
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import folium
from pathlib import Path

#set the project root
PROJECT_ROOT = Path.cwd().parent

#set data path
data_path = PROJECT_ROOT /  'strava.csv'

In [2]:
# set and view the data in a dataframe
df = pd.read_csv(data_path)
df

,Air Power,Cadence,Form Power,Ground Time,Leg Spring Stiffness,Power,Vertical Oscillation,altitude,cadence,datafile,...,enhanced_speed,fractional_cadence,heart_rate,position_lat,position_long,speed,timestamp,unknown_87,unknown_88,unknown_90
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,activities/2675855419.fit.gz,...,0.000,0.0,68.0,NaN,NaN,0.0,2019-07-08 21:04:03,0.0,300.0,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,activities/2675855419.fit.gz,...,0.000,0.0,68.0,NaN,NaN,0.0,2019-07-08 21:04:04,0.0,300.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.0,activities/2675855419.fit.gz,...,1.316,0.0,71.0,NaN,NaN,1316.0,2019-07-08 21:04:07,0.0,300.0,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3747.0,77.0,activities/2675855419.fit.gz,...,1.866,0.0,77.0,504432050.0,-999063637.0,1866.0,2019-07-08 21:04:14,0.0,100.0,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3798.0,77.0,activities/2675855419.fit.gz,...,1.894,0.0,80.0,504432492.0,-999064534.0,1894.0,2019-07-08 21:04:15,0.0,100.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40644,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.0,activities/2925939753.fit.gz,...,5.981,0.0,143.0,504554553.0,-999308618.0,NaN,2019-10-03 23:04:54,0.0,300.0,NaN
40645,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.0,activities/2925939753.fit.gz,...,4.115,0.0,142.0,504553919.0,-999309466.0,NaN,2019-10-03 23:04:56,0.0,300.0,NaN
40646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.0,activities/2925939753.fit.gz,...,3.322,0.0,142.0,504553588.0,-999309432.0,NaN,2019-10-03 23:04:57,0.0,300.0,NaN
40647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0,activities/2925939753.fit.gz,...,1.680,0.0,138.0,504552459.0,-999308808.0,NaN,2019-10-03 23:05:02,0.0,300.0,NaN


## Data Overview

The data set we are using is an export from Strava activity.  The export does not contain information on column meaning of what type of activity each datafile represents.  The following is a quick overview of the data set.

From the following data overview we can see many of the columns are missing a lot of data so we will ignore them for this analysis

**NOTE** A Full Schema with column metada did not come with this data set.  To understand the data I needed to go to Strava and search the internet to help me understand the columns. https://support.strava.com/en-us/articles/15401919-exporting-your-data-and-bulk-export

In [3]:

print('Data Overview')
print('-'*20)
#number of columns
num_cols = len(df.columns)
print(f'Number of Columns: {num_cols}')
print(f'Number of Rows: {len(df)}')
print(f'Number of Unique Activities: {df['datafile'].nunique()}')
print('Missing values in comlumns')
print(df.isna().sum())

Data Overview
--------------------
Number of Columns: 22
Number of Rows: 40649
Number of Unique Activities: 64
Missing values in comlumns
Air Power               22807
Cadence                 22802
Form Power              22807
Ground Time             22802
Leg Spring Stiffness    22807
Power                   22802
Vertical Oscillation    22802
altitude                25744
cadence                    22
datafile                    0
distance                    0
enhanced_altitude          51
enhanced_speed             10
fractional_cadence         22
heart_rate               2294
position_lat              192
position_long             192
speed                   25721
timestamp                   0
unknown_87                 22
unknown_88               2294
unknown_90              22031
dtype: int64


## Initial Data Exploration: Weekly Milage

The First thing I want to look at is the weekly milage the professor is doing.  This will give me some insight into how many runs and how far he has run for each workout.  This is the start of my exploration and is a basic visualization to get myself familiar with what the professor was doing.  Was he training for a race? Was he just casually exercising? Was his fitness improving?  What else was he doing.  

The data does not contain an activity type so my initial assumption is that he is running

In [4]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

runs_df = df.copy()

runs_df = df.groupby('datafile').agg(timestamp=('timestamp','first'), total_distance=('distance','max')).reset_index()

runs_df['week'] = runs_df['timestamp'].dt.to_period('W-MON').dt.start_time

In [5]:
fig = px.bar(
    runs_df,
    x='week',
    y='total_distance',
    color='datafile',
    title='Weekly Milage Breakdown by Run',
    labels={'week':'Week','total_distance':'Total Distance'}
)

fig.show()

### Chart Analysis

Looking at the chart, it appears the training was building up to a long-distance run at the end of September. The routine started with many short runs for the two months of July and August, then moved into longer-distance runs in September, more than doubling his weekly output. At the end, it looks like he ran a long-distance race.

Based on the Strava website https://support.strava.com/en-us/articles/15401893-how-distance-is-calculated (and other google searches), the units look to be in meters.  38,600 meters / 1,609 meters = 23.94 miles = ~24 miles

Without knowing what the professor actually did, my assumption is he was running a marathon (26.2) miles and one of a few things happened:
1. Strava miscalucated the milage (or the runner started tracking late)
2. The course was short
3. The runner Did Not Finish (DNF), possibly sustaining an injury


Reviewing https://www.brooksrunning.com/en_no/blog/training-workouts/how-long-does-it-take-to-train-for-a-marathon.html?srsltid=AfmBOorHJY9-QUclAb-8g2SvInxxuKboPipT6d3OfAE7ZQSeENf33kWI , it takes 16-20 weeks to train for a marathon.  The data only has <12 weeks of activity which may not be enough time to properly train for a marathon and the runner most likely DNF the race.  The previous run right before the race was just ~39,000 meters, which is very similar to the final run.  That would be running a practice marathon before the actual race so there is still a chance that 1 or 2 was the problem

But before we go any further, can we find some other info that will help us deterime if he was actually running a marathon?

The first step to do this is to check out the duration of the activity.  That will give us some clues

In [6]:
#find how long the user ran for
final_race = 'activities/2925939753.fit.gz'

df_final_race = df[df['datafile']==final_race].copy()

race_time = df_final_race['timestamp'].max()-df_final_race['timestamp'].min()

print(f'Total Race Time: {race_time}')


Total Race Time: 0 days 01:35:02


This is a huge finding and another clear reason why you need to look into your data carefully and also why assumptions are bad.

The world record for a marathon is 1:59:30. The probability of the professor beating that pace is as close to zero as you can get. Therefore, I can conclude one of two things:
1. He was not running a marathon at all.
2. My distance conversion is wrong (I checked and it's not).

After a Google search, I learned that ~25 mile bike rides are common and his pace seems ordinary for cycling, so my guess is his final event was a bike ride.

Let me rerun the previous chart with event duration so I can check whether the professor was running or biking.

In [7]:
runs_df = df.copy()

runs_df = df.groupby('datafile').agg(
    timestamp=('timestamp','first'), 
    total_distance=('distance','max'),
    event_duration=('timestamp',np.ptp)).reset_index()


runs_df['pace'] = runs_df['total_distance'] / (runs_df['event_duration'].dt.total_seconds() /60 /60)

runs_df['week'] = runs_df['timestamp'].dt.to_period('W-MON').dt.start_time



fig = px.bar(
    runs_df,
    x='week',
    y='total_distance',
    color='pace',
    title='Weekly Milage Breakdown by Activity',
    labels={'week':'Week','total_distance':'Total Distance'}
)

fig.show()

From the new visualization, which shows distance by week where each event is color-coded by pace, it appears the professor was either walking or running to begin with and then switched to something much faster. Based on the assignment information, which states he bought a bicycle, my assumption is he started biking in September after running in the summer.

### Performance improvement

Now that we can see what types of exercises he was doing (or at a minimum cluster them by pace) let's look to see if the professor was actually improving his fitness. We can do that by looking at a few things, such as heart rate changes throughout his workouts, how fast he came back to resting heart rate (if the data is available), speed, and cadence. I was not sure what the definition of cadence was, so I found it on https://support.strava.com/en-us/articles/15401948-cadence

So, what should we look at first?

Well, to do this I need to sepearte out the clusters since they are not relateable.  My assumptions are the professor was running before September and biking in September.

In [8]:
#creating a new field to measure fitness
df['candence_hr_ratio'] = df['cadence'] /df['heart_rate']

#creating two dataframes for the different types of exercises based on the data
df_runs = df[df['timestamp']<'2019-09-01'].copy()
df_bikes = df[df['timestamp']>='2019-09-01'].copy()

In [9]:
run_daily_efficiency = df_runs.groupby('datafile').agg(
    timestamp=('timestamp','first'),
    cadence_hr_ratio=('candence_hr_ratio','mean')).reset_index()

bike_daily_efficiency = df_bikes.groupby('datafile').agg(
    timestamp=('timestamp','first'),
    cadence_hr_ratio=('candence_hr_ratio','mean')).reset_index()

In [ ]:
#figure to see how efficient his runs are
fig1 = px.scatter(
    data_frame=run_daily_efficiency,
    x='timestamp',
    y='cadence_hr_ratio',
    trendline='ols', # Ordinary Least Squares best-fit line
    title=" Heart Rate vs. Cadence for Runs with Best fit Line"
)

#figure to see how efficient what I think his bike rides are
fig2 = px.scatter(
    data_frame=bike_daily_efficiency,
    x='timestamp',
    y='cadence_hr_ratio',
    trendline='ols', # Ordinary Least Squares best-fit line
    title=" Heart Rate vs. Cadence for Bikes with Best fit Line"

)

#chart to see how his heart rate changes over time across all the workouts.  
#Done to see if there is any improvements of the top and box ranges both going down over time
fig3 = px.scatter(data_frame=df,
        x='timestamp',
        y='heart_rate',
        opacity=0.6,
        title="Heart Rate Range During Workouts over Time")


fig1.show()
fig2.show()
fig3.show()

### Conclusion and Insights


#### Runs
From the data above, we can see a clear trend line from upper left to lower right in his heart rate vs. cadence for runs. Across many data points over two months, a clear trend of improving fitness emerges. It would be helpful to overlay average distance per run onto the graph to determine if he is maintaining or increasing his distance. Covering farther distances with an improving ratio would provide even stronger evidence of progress.

Based on the data in "Heart Rate Range During Workouts over Time," it appears the professor was pushing himself hard toward the end of August, which aligns with the data in "Heart Rate vs. Cadence per Run."

#### Bikes
There are only a few observation in the bikes data.  The best fit line is slightly sloped downward, but I do not think that is enough to tell us his fitness was improving.  It looks like he is more maintaining than anything else.


#### Insight
From the data above, we can see that running requires more energy and involves a wider range of heart rates, causing the body to work harder than biking. Although biking covers farther distances in the same amount of time, running seems to give the biggest bang for the buck to improve cardio. However, there are not enough bike data points to make this conclusive.


## Analysis: Workout Duration vs. Distance Over Time

From the previous analysis, we thought it would be valuable to examine whether the professor was covering greater distances as his heart rate and cadence improved, rather than simply running shorter routes. By analyzing workout duration alongside total distance over time, we can evaluate whether his lower heart rate reflects genuine cardiovascular gains, such as maintaining a higher pace over longer distances at a lower physiological cost

In [11]:

#update the new dataframe with duration and distance for plotting
run_daily_efficiency = df_runs.groupby('datafile').agg(
    timestamp=('timestamp','first'),
    total_distance=('distance','max'),
    cadence_hr_ratio=('candence_hr_ratio','mean'),
    event_duration=('timestamp',np.ptp)).reset_index()

run_daily_efficiency['event_duration'] = run_daily_efficiency['event_duration'].dt.total_seconds() / 60

run_daily_efficiency = run_daily_efficiency.sort_values('timestamp')

# print(run_daily_efficiency)

fig = make_subplots(specs=[[{"secondary_y":True}]])

#add the first Scatter plot with lines to the graphs as a trace in plotly
fig.add_trace(
    go.Scatter(
        x=run_daily_efficiency['timestamp'],
        y=run_daily_efficiency['cadence_hr_ratio'],
        name='Efficiency Ratio',
        mode='lines',
        # line='ols'
    ),
    secondary_y=False
)

#add the second scatter plot without lines to the graph as a trace in plotly to the second y axis
fig.add_trace(
    go.Scatter(
        x=run_daily_efficiency['timestamp'],
        y=run_daily_efficiency['total_distance'],
        name="Distance",
        mode='markers',
        line=dict(color='darkorange', width=2),
        marker=dict(
            color=run_daily_efficiency['event_duration'],
            colorscale='Viridis',
            size=8,
            showscale=True,
            colorbar=dict(title="Duration<br>(mins)")
        )
    ),
    secondary_y=True
)


# 4. Format the layout and axis titles
fig.update_layout(
    title_text="Workout Duration vs. Distance Over Time",
    hovermode="x unified", # Consolidates hover info for both metrics into one box
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Label the left y-axis
fig.update_yaxes(title_text="<b>Efficiency Ratio", secondary_y=False)

# Label the right y-axis
fig.update_yaxes(title_text="<b>Distance</b> (Meters)", secondary_y=True)

fig.show()



### Insights

From the previous analysis, we wanted to verify whether the professor was extending his distance over time rather than just running shorter, easier routes. 

Looking at the data from July through August 2019, two distinct patterns emerge:

- **Training Volume & Consistency:** In early July, run distances and durations fluctuated widely between 1.5km and 10km. By August, his routine stabilized into a consistent training block—regularly covering 5k–6k distances within a 40–50 minute duration window, supplemented by shorter recovery efforts.
- **Efficiency Trend:** The Efficiency Ratio peaked in late July (~0.74) before declining through August to around ~0.51–0.57. This drop directly aligns with the period he was pushing himself harder, demonstrating that sustained effort over consistent distances exerted greater strain during this training phase.

# Event Details, Lets see how any one of the professors workouts looks

## Interactive Dashboard: Comparing Route Terrain & Physical Effort

To truly understand the professor's performance, I do not think looking at averages is enough.  We need to see how he responds to real-world terrain on a segment-by-segment level during a run or a ride. 

I built this interactive, side-by-side dashboard to analyze every workout across his entire history from start to finish:

- **3D Terrain & Heart Rate Profile (Left):** Maps the route in 3D (latitude, longitude, and altitude) colored by **Heart Rate**. I originally had speed here, but switching this from speed to heart rate removes redundancy with the second graph and reveals how strain shifts with elevation. It now shows how his heart rate responds to climbing uphill versus recovering downhill.
- **Street-Level Map & Speed Profile (Right):** Shows the exact street route color-coded by **Speed**, providing real-world context so we can see where he actually accelerated, slowed down, or tackled specific roads.

Having these views side-by-side allows us to evaluate the direct relationship between speed, elevation, and heart rate for any given event. By cycling through every workout in the dataset, we can observe how his pacing and cardiovascular response evolved over time across different routes.

In [12]:
# all Races
races = df['datafile'].unique().tolist()

# create dropdown to select race
dropdown = widgets.Dropdown(options=races, description='Select Event:')

# create the layout
fig_widget = go.FigureWidget(
    layout=go.Layout(
        height=500,
        margin=dict(l=0, r=20, t=40, b=0),
        scene=dict(
            domain=dict(x=[0,0.85]), # leave some room for the legend
            xaxis_title='lat',
            yaxis_title='lon',
            zaxis_title='enhanced_altitude'
        ),
        showlegend=True
    )
)
# let HBox control sizing
fig_widget.layout.width = None  

# swap HTML widget for Output widget to bypass CSP tile blocking
map_out = widgets.Output(layout=widgets.Layout(width='100%', height='600px'))
map_title = widgets.HTML(value="")

#create a function to dynamically change between activities
def update_dashboard(event_id):
    selected_event = df[df['datafile'] == event_id].copy()
    selected_event = selected_event.dropna(subset=['position_lat', 'position_long'])

    if selected_event.empty:
        with fig_widget.batch_update():
            fig_widget.data = []
            fig_widget.layout.title = 'No data for this event'
        with map_out:
            clear_output(wait=True)
        map_title.value = ''
        return

    selected_event['lat'] = selected_event['position_lat'] * (180 / 2**31)
    selected_event['lon'] = selected_event['position_long'] * (180 / 2**31)

    # create and update the 3D scatter plot
    with fig_widget.batch_update():
        fig_widget.data = []
        fig_widget.add_scatter3d(
            x=selected_event['lat'],
            y=selected_event['lon'],
            z=selected_event['enhanced_altitude'],
            mode='markers',
            marker=dict(
                color=selected_event['heart_rate'],
                colorscale='Plasma',
                showscale=True,
                colorbar=dict(
                    title='Heart Rate',
                    x=0.81,
                    xanchor='left',
                    y=0.35,
                    thickness=12)
            ),
            name='route',
            showlegend=True
        )
        fig_widget.layout.title = f'3D Terrain Profile: {event_id}'

    # update map title
    map_title.value = f"<h3 style='text-align: center; margin-top: 10px;'>Map View: {event_id}</h3>"
    
    start_lat = selected_event['lat'].iloc[0]
    start_lon = selected_event['lon'].iloc[0]

    speed_min = selected_event['enhanced_speed'].min()
    speed_max = selected_event['enhanced_speed'].max()

    # create a named colormap that renders a legend
    colormap = folium.branca.colormap.LinearColormap(
        colors=['blue', 'cyan', 'green', 'yellow', 'red'],
        vmin=speed_min,
        vmax=speed_max,
        caption='Speed'
    )

    # create the map
    m = folium.Map(location=[start_lat, start_lon], tiles='CartoDB positron')
    folium.ColorLine(
        positions=list(zip(selected_event['lat'], selected_event['lon'])),
        colors=selected_event['enhanced_speed'].tolist(),
        colormap=colormap,
        weight=5,
        opacity=0.8
    ).add_to(m)

    colormap.add_to(m)

    min_lat, max_lat = selected_event['lat'].min(), selected_event['lat'].max()
    min_lon, max_lon = selected_event['lon'].min(), selected_event['lon'].max()
    m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    # Strictly clear and display the map inside the output container
    with map_out:
        clear_output(wait=True)
        display(m)

# wire dropdown
def on_change(change):
    if change['name'] == 'value':
        update_dashboard(change['new'])

dropdown.observe(on_change, names='value')

# Update the row layout
row_layout = widgets.Layout(
    display='flex',
    flex_flow='row nowrap',
    justify_content='space-between',
    width='100%'
)

left_box  = widgets.Box([fig_widget], layout=widgets.Layout(width='48%', height='600px'))
right_box = widgets.VBox([map_title, map_out], layout=widgets.Layout(width='48%', height='600px'))

ui = widgets.VBox([
    dropdown,
    widgets.HBox([left_box, right_box], layout=row_layout)
])

# Display the main UI exactly once to prevent duplication
display(ui)

# render initial state
update_dashboard(dropdown.value)

# Conclusion

The Strava dataset shows two distinct phases: an earlier period of lower-speed running and a later period consistent with cycling. The longest activity is much too fast to be a marathon run, and its pace is consistent with a bike ride. Run fitness appears to improve over time based on cadence/heart-rate trends, while the bike data is too limited to conclude any fitness improvements. 

Evidence:

- Weekly distance grows, but duration and pace show a switch in mode around September.
- The final activity duration is far longer than expected for a marathon at realistic running speed.
- Heart rate vs. cadence trends indicate improving fitness for the earlier “run” phase.
- The bike-phase dataset is smaller, so conclusions about bike fitness are less certain.